# Retrieve the vouchers from the url and code

In [1]:
# autoreload must be loaded before project imports
%load_ext autoreload
%autoreload 2

# python modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json
import os
from dotenv import load_dotenv
from datetime import datetime

import pandas as pd
pd.set_option('display.max_columns', None)


# our modules:
import sys
# sys.path.insert(0, '/content/my_utils') # this is for BigQuery
# sys.path.append('../../src') # this is for running locally
from utils_test.my_module import my_sum
from utils_test.my_second_module import my_sum_3
# Import your centralized path configuration
from configs.settings import work_dir
from proton_email.login import ProtonEmailLogin, LibraVoucherDetail
from priceless.retrieve_vouchers import RetrieveVouchers, export_combined_vouchers_to_excel

# From Proton Mail get the voucher url and code first and for all retrieve

In [2]:
prefix = "tati"
# since = None
since = datetime(2026, 6, 21, 0, 0)
proton_debug_version = "v18"
retrieve_debug_version = "v18"

## Get voucher details from Proton email

In [4]:

proton = ProtonEmailLogin(
    debug=True, 
    debug_version=proton_debug_version,
    headless = True,
    # headless=False,  # only for the first login / CAPTCHA
)
try:
    details = await proton.get_libra_voucher_details(since=since)
finally:
    await proton.close()
details

Launching browser (headless=True, debug=True)...
Screenshots -> /opt/priceless/data/debug/proton_email
Loading saved Proton session from /opt/priceless/data/proton_email/sessions/stefan.buzatu21_at_proton.me.json
Inbox URL: https://mail.proton.me/u/0/inbox
Reused saved Proton session.
Screenshot: /opt/priceless/data/debug/proton_email/v18_01_authenticated.png
Opening inbox...
Inbox URL: https://mail.proton.me/u/0/inbox
Screenshot: /opt/priceless/data/debug/proton_email/v18_02_inbox.png
Looking for "Detalii comand* e-voucher" thread...
Waiting for message list...
Message list ready via .item-container:not(.item-is-loading)
Scanning 11 inbox rows for subject match...
Matched email: Detalii comandă e-voucher
Opened voucher thread: Detalii comandă e-voucher
Reading 5 message(s) one by one...
Expanding all messages in thread...
Found 7 message iframe(s).
Screenshot: /opt/priceless/data/debug/proton_email/v18_03_voucher_msg_01.png
  extracted comanda#=979176 redemption_id=1008541727 code=227

[LibraVoucherDetail(url='https://rl.rewardcloud.io/index/452f5506-5af2-44c6-889d-13ee5915f935', code='227e66c6-2c89-48b6-b3b8-7a0bb11ba1eb', received_at='Sunday, 21 June 2026 at 4:46 AM', comanda_nr='979176', redemption_id='1008541727', comanda_id='979176-2', cantitate='2', puncte='221', subtotal='442', total='442', received_at_dt=datetime.datetime(2026, 6, 21, 4, 46)),
 LibraVoucherDetail(url='https://rl.rewardcloud.io/index/6d7f693f-e127-4b65-baab-36224ee7aa67', code='601b709c-73ea-4209-ab3a-7384399d76f2', received_at='Sunday, 21 June 2026 at 4:46 AM', comanda_nr='979176', redemption_id='1008541726', comanda_id='979176-1', cantitate='7', puncte='1.290', subtotal='9.030', total='9.030', received_at_dt=datetime.datetime(2026, 6, 21, 4, 46)),
 LibraVoucherDetail(url='https://rl.rewardcloud.io/index/d90d6134-196e-402e-8045-62bd11af0ac2', code='6767e4e7-b9f4-408e-a57d-7a18da17ee72', received_at='Sunday, 21 June 2026 at 9:51 PM', comanda_nr='979649', redemption_id='1008606829', comanda_id=

## Retrieve vouchers and store each in an Excel file

In [9]:
excel_paths = []
results = []

for index, detail in enumerate(details, start=1):
    retriever = RetrieveVouchers(
        detail,
        prefix=prefix,
        debug=True,
        debug_version="%s_%02d" % (retrieve_debug_version, index),
    )
    try:
        result = await retriever.get_vouchers()
        results.append(result)
        excel_paths.append(retriever._last_excel_path)
    finally:
        await retriever.close()

combined_path = export_combined_vouchers_to_excel(results, prefix=prefix) if results else None
if combined_path:
    print("Combined Excel saved:", combined_path)

excel_paths

Launching browser (headless=True, debug=True)...
Screenshots -> /opt/priceless/data/debug/priceless_retrieve_vouchers
Opening https://rl.rewardcloud.io/index/452f5506-5af2-44c6-889d-13ee5915f935
Screenshot: /opt/priceless/data/debug/priceless_retrieve_vouchers/v18_01_01_code_page.png
Screenshot: /opt/priceless/data/debug/priceless_retrieve_vouchers/v18_01_02_code_filled.png
Screenshot: /opt/priceless/data/debug/priceless_retrieve_vouchers/v18_01_03_after_continue.png
Screenshot: /opt/priceless/data/debug/priceless_retrieve_vouchers/v18_01_04_vouchers_loaded.png
Screenshot: /opt/priceless/data/debug/priceless_retrieve_vouchers/v18_01_05_vouchers_retrieved.png
Retrieved 2 gift card(s).
  card 1 id=6622432 code=8567-9018-3744-2130 expires=2027-06-13 (EMAG Romania - 20.00 RON)
  card 2 id=6622433 code=3781-7697-2727-9096 expires=2027-06-13 (EMAG Romania - 20.00 RON)
Excel saved: /opt/priceless/data/priceless_folder/tati_2026-06-21-04-46-00_979176-2_1008541727.xlsx
Launching browser (headle

[PosixPath('/opt/priceless/data/priceless_folder/tati_2026-06-21-04-46-00_979176-2_1008541727.xlsx'),
 PosixPath('/opt/priceless/data/priceless_folder/tati_2026-06-21-04-46-00_979176-1_1008541726.xlsx'),
 PosixPath('/opt/priceless/data/priceless_folder/tati_2026-06-21-21-51-00_979649-2_1008606829.xlsx'),
 PosixPath('/opt/priceless/data/priceless_folder/tati_2026-06-21-21-51-00_979649-1_1008606828.xlsx'),
 PosixPath('/opt/priceless/data/priceless_folder/tati_2026-06-21-21-51-00_979649-3_1008606830.xlsx')]

In [6]:
# excel_paths = []
# results = []

# for index, detail in enumerate(details, start=1):
#     retriever = RetrieveVouchers(
#         detail,
#         prefix=prefix,
#         debug=True,
#         debug_version="%s_%02d" % (retrieve_debug_version, index),
#     )
#     try:
#         result = await retriever.get_vouchers()
#         results.append(result)
#         excel_paths.append(retriever._last_excel_path)
#     finally:
#         await retriever.close()

# combined_path = export_combined_vouchers_to_excel(results, prefix=prefix) if results else None
# if combined_path:
#     print("Combined Excel saved:", combined_path)

# excel_paths

Launching browser (headless=True, debug=True)...
Screenshots -> /opt/priceless/data/debug/priceless_retrieve_vouchers
Opening https://rl.rewardcloud.io/index/452f5506-5af2-44c6-889d-13ee5915f935
Screenshot: /opt/priceless/data/debug/priceless_retrieve_vouchers/v18_01_01_code_page.png
Screenshot: /opt/priceless/data/debug/priceless_retrieve_vouchers/v18_01_02_code_filled.png
Screenshot: /opt/priceless/data/debug/priceless_retrieve_vouchers/v18_01_03_after_continue.png
Screenshot: /opt/priceless/data/debug/priceless_retrieve_vouchers/v18_01_04_vouchers_loaded.png
Screenshot: /opt/priceless/data/debug/priceless_retrieve_vouchers/v18_01_05_vouchers_retrieved.png
Retrieved 2 gift card(s).
  card 1 id=6622432 code=8567-9018-3744-2130 expires=2027-06-13 (EMAG Romania - 20.00 RON)
  card 2 id=6622433 code=3781-7697-2727-9096 expires=2027-06-13 (EMAG Romania - 20.00 RON)
Excel saved: /opt/priceless/data/priceless_folder/tati_2026-06-21-04-46-00_979176-2_1008541727.xlsx
Launching browser (headle

[PosixPath('/opt/priceless/data/priceless_folder/tati_2026-06-21-04-46-00_979176-2_1008541727.xlsx'),
 PosixPath('/opt/priceless/data/priceless_folder/tati_2026-06-21-04-46-00_979176-1_1008541726.xlsx'),
 PosixPath('/opt/priceless/data/priceless_folder/tati_2026-06-21-21-51-00_979649-2_1008606829.xlsx'),
 PosixPath('/opt/priceless/data/priceless_folder/tati_2026-06-21-21-51-00_979649-1_1008606828.xlsx'),
 PosixPath('/opt/priceless/data/priceless_folder/tati_2026-06-21-21-51-00_979649-3_1008606830.xlsx')]

In [ ]:
# # 2) Retrieve gift-card codes from Reward Cloud
# retriever = RetrieveVouchers(
#     details[0],
#     debug=True,
#     debug_version="v01",
# )
# result = await retriever.get_vouchers()
# await retriever.close()

In [ ]:
# voucher = LibraVoucherDetail(
#     url="https://rl.rewardcloud.io/index/67a140ae-5136-4879-ad50-1b0b9c2ac5c5",
#     code="32ce7b7c-3114-4108-b6d6-709585a1d0b8",
#     received_at="",
# )
# voucher

In [ ]:
assert False

# This is running once

In [ ]:
voucher = LibraVoucherDetail(
    url="https://rl.rewardcloud.io/index/676c450d-f1d5-4a33-9ff3-302dc6356c90",
    code="7714061d-cf10-4303-b6b2-516c1b831204",
    received_at="Sunday, 21 June 2026 at 4:49 AM",
    comanda_nr="979182",
    redemption_id="1008541841",
    comanda_id="979182-1",
    cantitate="3",
    puncte="1.290",
    subtotal="3.870",
    total="3.870",
    received_at_dt=datetime(2026, 6, 21, 4, 49),
)
voucher

In [ ]:
retriever = RetrieveVouchers(
    voucher,
    debug=True,
    debug_version="v05",
    prefix="adi",
)
retriever

In [ ]:
retriever.excel_output_path()  # preview path before running

In [ ]:
result = await retriever.get_vouchers()
result